# sys1 v3: fast encoder training, on GPU

Same script as running locally (`training/train_fast_encoder.py`) -- it's
fully self-contained (no dependency on the rest of the `sys1` package) and
already auto-detects CUDA, so this notebook just needs the script itself
plus the data files.

This exists because the local CPU run hit a real, confirmed issue: pinning
to a single CPU thread (needed to avoid an intermittent native segfault
under PyTorch's CPU thread pool) makes a 20k-example, 6-epoch run take many
hours. A GPU run doesn't share that CPU-specific thread-pool contention and
should finish in minutes.

Steps:
1. Runtime > Change runtime type > GPU (T4 is fine).
2. Upload `training/train_fast_encoder.py` and the four `data/*.jsonl`
   files (`seed_examples.jsonl`, `real_examples.jsonl`,
   `benchmark_domain_examples.jsonl`, `bev_decision_examples.jsonl`) to
   this Colab session's `/content/`.
3. Run all cells.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# Larger batch size than the local CPU run (32 vs 16) -- a GPU has plenty
# of headroom for these tiny sequences, and bigger batches mean fewer,
# more efficient steps.
!python /content/train_fast_encoder.py \
  /content/seed_examples.jsonl \
  /content/real_examples.jsonl \
  /content/benchmark_domain_examples.jsonl \
  /content/bev_decision_examples.jsonl \
  --batch-size 32 \
  --out-dir /content/sys1-fast-encoder-v3

In [ ]:
!zip -r /content/sys1-fast-encoder-v3.zip /content/sys1-fast-encoder-v3
# then download sys1-fast-encoder-v3.zip and extract into models/ locally.